In [ ]:
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
import matplotlib.pyplot as plt

from emu_renewal.inputs import OUTPUTS_PATH, get_world_shp
from emu_renewal.outputs import get_prop_improve, get_param_vals_by_analysis
from emu_renewal.plotting import MOB_SOURCE_MAP, MOB_COLOURS, ANALYSIS_TYPES

In [ ]:
plt.style.use("default")
set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
countries = ls(job_path)
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1)
category_colors = {
    "no_mob": "0.45",
    "g_mob": "green",
    "a_mob": "red",
    "fb_mob": "blue",
}

In [ ]:
param = "dispersion_proc"
posts = {}
for c in countries:
    posts[c] = get_param_vals_by_analysis(param, job_path / c)
    world.loc[world["ISO_A3"] == c, "best_mob"] = posts[c].mean().idxmin()
    world["best_mob_colour"] = world["best_mob"].map(category_colors)
world.loc[world["best_mob_colour"].isna(), "best_mob_colour"] = "0.975"

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 7), constrained_layout=True)
axes = axs.ravel()

# Common features
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    world.boundary.plot(ax=ax, color="black", linewidth=0.2)

# Best mobility map
ax = axes[0]
ax.set_title("Best mobility type")
world.plot(ax=ax, color=world["best_mob_colour"])

# Proportion of runs for which mobility type improves fit
for a, mob_type in enumerate(ANALYSIS_TYPES[1:]):
    ax = axes[a + 1]
    ax.set_title(MOB_SOURCE_MAP[mob_type])

    prop_improve = {}
    for c in countries:
        c_posts = posts[c]
        if mob_type in c_posts:
            c_posts[mob_type] = np.random.permutation(c_posts[mob_type].values)
            prop_improve[c] = pd.Series(c_posts["no_mob"] > c_posts[mob_type]).astype(int).mean()
    mob_str = f"{mob_type}_improve"
    world[mob_str] = world["ISO_A3"].map(prop_improve)
    world.plot(column=mob_str, ax=ax, cmap="coolwarm", legend=True, missing_kwds={"color": "0.975"}, vmin=0.0, vmax=1.0)

In [ ]:
fig.savefig("map_results.pdf")